# Entrenamiento del detector en Colab (v7: YOLO11m sobre el conjunto v5)

Antes de ejecutar:
1. Menú **Entorno de ejecución > Cambiar tipo de entorno > GPU** (A100 o L4 si hay plan de pago; T4 en el plan gratuito, que es más lenta que la RTX 5070 Ti local).
2. Subir `datasets_v5.zip` a Drive en `MyDrive/tesis_yolo/`.
3. Ejecutar las celdas en orden. Los resultados se guardan en `MyDrive/tesis_yolo/runs/`, así que si la sesión se corta se retoma con la última celda.

Mismos hiperparámetros que los entrenamientos locales (`train_detector_v2.py`): semilla 0, cierre de mosaico en las últimas 10 épocas, el resto por defecto de Ultralytics.

In [ ]:
# 1. GPU y Drive
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Ultralytics (misma versión que el entrenamiento local)
!pip install -q ultralytics==8.4.115
import ultralytics; print(ultralytics.__version__)

In [ ]:
# 3. Descomprimir el conjunto en el disco local de Colab (mucho más rápido que leer desde Drive)
import os, zipfile
ZIP = '/content/drive/MyDrive/tesis_yolo/datasets_v5.zip'
DEST = '/content/datasets_v5'
if not os.path.exists(DEST + '/data.yaml'):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(DEST)
yaml_text = (
    'path: ' + DEST + '\n'
    'train: images/train\nval: images/val\ntest: images/test\n\n'
    'names:\n  0: person\n  1: car\n  2: motorcycle\n  3: fire\n  4: smoke\n  5: accident\n'
)
open(DEST + '/data.yaml', 'w').write(yaml_text)
for s in ('train', 'val', 'test'):
    print(s, len(os.listdir(f'{DEST}/images/{s}')), 'imágenes')

In [ ]:
# 4. Entrenar. Cambiar MODEL, IMGSZ y NAME para otras variantes (ej. yolo26m.pt, imgsz 960).
import os
from ultralytics import YOLO
MODEL = 'yolo11m.pt'
IMGSZ = 640
BATCH = 16          # con A100 (40 GB) se puede subir a 32; con T4 bajar a 8 si falta memoria
EPOCHS = 40
PATIENCE = 8
NAME = 'baseline_v7_training'
PROJECT = '/content/drive/MyDrive/tesis_yolo/runs'

model = YOLO(MODEL)
model.train(
    data='/content/datasets_v5/data.yaml', epochs=EPOCHS, patience=PATIENCE, batch=BATCH, imgsz=IMGSZ,
    workers=min(8, os.cpu_count() or 2), device=0, project=PROJECT, name=NAME, exist_ok=True,
    seed=0, close_mosaic=10, plots=True,
)

In [ ]:
# 5. Si la sesión se cortó, retomar desde la última época guardada (volver a ejecutar las celdas 1 a 3 primero)
from ultralytics import YOLO
YOLO('/content/drive/MyDrive/tesis_yolo/runs/baseline_v7_training/weights/last.pt').train(resume=True)

Al terminar, descargar de Drive `runs/baseline_v7_training/weights/best.pt` y `results.csv` y copiarlos a `monitoreo_urbano_app/data/models/yolo/baseline_v7.pt` y a `baseline_v7_training/results.csv`. La evaluación (umbrales en validación, prueba, COCO y videos) se corre en la máquina local con `scripts/evaluate_candidate.py --tag v7`.